<a href="https://colab.research.google.com/github/bsheese/387/blob/main/19_onramp/19_1_ML_Foundations/19_1_2_Confusion_Matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The On-Ramp: Part 2
## The Confusion Matrix

---

## What This Notebook Is About

Accuracy counts how many predictions are right, but it doesn't say which ones are wrong or what they were mistaken for. Our pet-breed model's 0.885 could mean an 11.5% error rate spread evenly over all four breeds, or a model that is perfect on three breeds and weak on the fourth. Those are very different models, and which one you have matters as soon as some mistakes cost more than others.

This notebook breaks the model's test accuracy down:

1. The **confusion matrix**: a table of every true breed against every predicted breed
2. **Precision** and **recall**: two numbers that describe how the model does on one breed
3. The **accuracy paradox**: a case where a useless model and a useful one have almost the same accuracy

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_1_1](https://github.com/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_1_Classification_Foundations.ipynb), [18_1_2](https://github.com/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_2_Confusion_Matrix_Basic_Metrics.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

---

## Section 1: Counting Every Kind of Mistake

The next cell loads the model's saved predictions, as in 19_1_1, and keeps the test photos.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    # It saves to a temporary name first, so a download that fails partway doesn't leave a broken file behind.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name + '.part')
        os.replace(name + '.part', name)
    return name

preds = pd.read_csv(fetch('pets4_predictions.csv'))
BREEDS = ['Birman', 'Ragdoll', 'basset_hound', 'beagle']

is_test = preds['split'].to_numpy() == 'test'
true_breed = preds['breed'].to_numpy()[is_test]
predicted = preds['predicted'].to_numpy()[is_test]
print(len(true_breed), 'test photos')
print(f'accuracy: {(predicted == true_breed).mean():.3f}')

A confusion matrix has one row for each true breed and one column for each predicted breed. Each cell counts the photos with that combination. We can build one ourselves: start with a 4 × 4 array of zeros, then go through the test photos one at a time and add 1 to the cell for that photo's true breed (the row) and predicted breed (the column).

`np.zeros((4, 4), dtype=int)` makes the empty table. A two-dimensional array is indexed with a row and a column, so `matrix[i, j]` is the cell in row `i`, column `j`.

In [ ]:
matrix = np.zeros((4, 4), dtype=int)
for t, p in zip(true_breed, predicted):
    row = BREEDS.index(t)
    col = BREEDS.index(p)
    matrix[row, col] += 1

print(matrix)

scikit-learn has a function that does the same counting, and another that draws the result with labels. The `labels` argument fixes the order of the rows and columns.

In [ ]:
matrix = confusion_matrix(true_breed, predicted, labels=BREEDS)
print(matrix)

ConfusionMatrixDisplay(matrix, display_labels=BREEDS).plot(cmap='Blues')
plt.xlabel('predicted breed')
plt.ylabel('true breed')
plt.title('Pets model, 200 test photos')
plt.show()

How to read it:

- **Each row** is one true breed, and its numbers add up to the 50 test photos of that breed. The Ragdoll row says that of the 50 Ragdolls, 15 were called Birmans and 35 were called Ragdolls.
- **Each column** is one predicted breed, and its numbers add up to how many times the model gave that answer. The Birman column says the model said "Birman" 63 times: 48 times for real Birmans and 15 times for Ragdolls.
- **The diagonal**, from top left to bottom right, holds the right answers. Everything off the diagonal is a mistake, and its position tells you what was mistaken for what.

The matrix shows things the accuracy can't. All the mistakes are between the two cats or between the two dogs. Most of them are Ragdolls called Birmans; the reverse mistake happens only twice. Only one beagle is missed, and it is called a basset hound.

Accuracy can be read off the matrix: it's the sum of the diagonal divided by the total. `np.diag` takes out the diagonal of a two-dimensional array.

In [ ]:
print('diagonal:', np.diag(matrix))
print(f'accuracy: {np.diag(matrix).sum() / matrix.sum():.3f}')

---

## Section 2: One Breed at a Time: Precision and Recall

Often one class matters more than the others. Suppose the model is used by an adoption website to tag photos of Ragdolls, a breed people often search for. For that job the only question is "Ragdoll or not?", and every test photo falls into one of four groups:

| | model says Ragdoll | model says something else |
|---|---|---|
| **really a Ragdoll** | **true positive (TP)**: found | **false negative (FN)**: missed |
| **not a Ragdoll** | **false positive (FP)**: false alarm | **true negative (TN)**: correctly left out |

All four counts come from the confusion matrix. Ragdoll is row 1 and column 1 (counting from 0).

In [ ]:
k = BREEDS.index('Ragdoll')

TP = matrix[k, k]                   # Ragdolls called Ragdoll
FN = matrix[k, :].sum() - TP        # rest of the Ragdoll row: Ragdolls called something else
FP = matrix[:, k].sum() - TP        # rest of the Ragdoll column: other breeds called Ragdoll
TN = matrix.sum() - TP - FN - FP    # everything else
print(f'TP = {TP}, FN = {FN}, FP = {FP}, TN = {TN}')

`matrix[k, :]` is row `k` (the `:` means "every column") and `matrix[:, k]` is column `k`.

Two numbers summarize how the model does on Ragdolls:

- **Recall** = TP / (TP + FN): of the real Ragdolls, what fraction did the model find? It's computed along the Ragdoll **row**.
- **Precision** = TP / (TP + FP): when the model says "Ragdoll", what fraction of the time is it right? It's computed down the Ragdoll **column**.

In [ ]:
recall = TP / (TP + FN)
precision = TP / (TP + FP)
print(f'Ragdoll recall:    {recall:.3f}   (found {TP} of {TP + FN} Ragdolls)')
print(f'Ragdoll precision: {precision:.3f}   ({TP} of the {TP + FP} photos it called Ragdoll were Ragdolls)')

For Ragdolls the model's precision is high and its recall is lower. When it says "Ragdoll" it is usually right, but it misses almost a third of the real Ragdolls (15 of 50), because it calls them Birmans.

Birmans are the mirror image. The 15 Ragdolls that were called Birmans are false positives for Birman, which lowers Birman's precision, while Birman's recall stays high. `classification_report` computes precision and recall for every breed at once. (It also prints the **F1 score**, which combines precision and recall into one number, and the **support**, which is the number of test photos of each breed.)

In [ ]:
print(classification_report(true_breed, predicted, labels=BREEDS, digits=3))

Each row of the report treats one breed as "the positive class" and every other breed as "not it". Notice that the same mistake shows up twice: a Ragdoll called a Birman lowers Ragdoll's recall *and* Birman's precision.

---

## Section 3: The Accuracy Paradox

The test set has 50 photos of each breed, but suppose the adoption site's photos look more like real shelters, where Ragdolls are rare. The next cell builds a test set like that from our test photos: all 150 photos of the other three breeds, and only the first 5 Ragdolls. `N_RAGDOLLS` sets how many Ragdolls go in.

In [ ]:
N_RAGDOLLS = 5                                    # how many Ragdolls the shelter set gets (up to 50)

is_ragdoll = true_breed == 'Ragdoll'
ragdoll_positions = np.where(is_ragdoll)[0]       # positions of the Ragdoll photos
keep = ~is_ragdoll                                # every photo that isn't a Ragdoll...
keep[ragdoll_positions[:N_RAGDOLLS]] = True       # ...plus the first N_RAGDOLLS Ragdolls

shelter_true = true_breed[keep]
shelter_pred = predicted[keep]
print(len(shelter_true), 'photos, of which', (shelter_true == 'Ragdoll').sum(), 'are Ragdolls')

Two new NumPy operations appear here. `np.where(mask)[0]` gives the positions where a mask is `True`, and `~` flips a mask, turning every `True` into `False` and every `False` into `True`.

The website only asks "Ragdoll or not?". Compare two Ragdoll detectors on this shelter set:

- **Our model**: say "Ragdoll" whenever the model predicts Ragdoll.
- **The lazy detector**: never say "Ragdoll". It doesn't look at the photo at all.

In [ ]:
really = shelter_true == 'Ragdoll'
detectors = {
    'our model': shelter_pred == 'Ragdoll',
    'lazy detector': np.zeros(len(shelter_true), dtype=bool),   # all False
}
for name, says in detectors.items():
    accuracy = (says == really).mean()
    TP = (says & really).sum()
    print(f'{name:14}  accuracy {accuracy:.3f}   found {TP} of {really.sum()} Ragdolls   false alarms {(says & ~really).sum()}')

(`says & really` is `True` only where both arrays are `True`, which is where the detector said "Ragdoll" and the photo really is one.)

The two detectors have almost the same accuracy, 0.981 and 0.968. The lazy detector is right about every one of the 150 other photos and wrong about all 5 Ragdolls, so it makes 5 mistakes. Our model finds 4 of the 5 Ragdolls, but it also gives 2 false alarms (Birmans it calls Ragdolls), so it makes 3 mistakes. The difference between the two is 2 photos out of 155.

Accuracy barely tells these two detectors apart, but one of them is useful and the other never finds a Ragdoll. This is the **accuracy paradox**: when one class is rare, a model that ignores it completely can still have very high accuracy, because almost every answer of "not that class" is right. The baseline from 19_1_1, always giving the most common answer, is exactly the lazy detector, and here it scores 0.968.

Recall exposes the difference immediately: 0.8 for our model and 0 for the lazy detector. Whenever one class is rare, look at precision and recall for that class, not just at accuracy.

---

## Summary

- A **confusion matrix** has a row for every true class and a column for every predicted class. The diagonal holds the right answers, and each off-diagonal cell shows what was mistaken for what. For the Pets model, every mistake is between look-alike breeds, and most are Ragdolls called Birmans.
- For one class, every prediction is a **true positive**, **false positive**, **false negative** or **true negative**.
- **Recall** = TP / (TP + FN), read along the class's row: what fraction of that class the model found. **Precision** = TP / (TP + FP), read down the class's column: how often the model is right when it gives that answer. The model's Ragdoll recall is 0.70 and its precision is 0.946.
- The **accuracy paradox**: when one class is rare, a model that never predicts it can have high accuracy. On a set with 5 Ragdolls in 155 photos, a detector that never says "Ragdoll" has almost the same accuracy as our model (0.968 against 0.981).
- NumPy used here: two-dimensional arrays (`matrix[i, j]`, `matrix[k, :]`, `matrix[:, k]`), `np.diag`, `np.where`, `~` to flip a mask and `&` to combine masks.

## What's next

The confusion matrix, precision and recall describe what the model gets right and wrong, but they treat the model as a box that turns photos into answers. 19_1_3 opens up the last step of that box. It shows that the layer that turns each photo's 512 features into four breed scores is doing a simple calculation, a weighted sum of the features for each breed, which is the same calculation as a single artificial neuron and as logistic regression.